In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv")

In [4]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
data.shape

(50000, 2)

In [6]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [7]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [8]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [9]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [10]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [11]:
train_data.shape

(40000, 2)

In [12]:
test_data.shape

(10000, 2)

In [13]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data["review"])

In [14]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=200)

In [15]:
X_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [16]:
X_test

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [17]:
Y_train = train_data["sentiment"]
Y_test = train_data["sentiment"]

In [18]:
Y_train

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [19]:
Y_test

,sentiment
39087,0
30893,0
45278,1
16398,0
13653,0
...,...
11284,1
44732,1
38158,0
860,1


In [20]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim= 128, input_length= 200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout= 0.2))
model.add(Dense(1, activation="sigmoid"))

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [23]:
model.fit(X_train, Y_train, epochs=5, batch_size= 64, validation_split= 0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 202s 389ms/step - accuracy: 0.7120 - loss: 0.5445 - val_accuracy: 0.8375 - val_loss: 0.3705
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 201s 403ms/step - accuracy: 0.8538 - loss: 0.3530 - val_accuracy: 0.8508 - val_loss: 0.3493
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 195s 390ms/step - accuracy: 0.8818 - loss: 0.2939 - val_accuracy: 0.8569 - val_loss: 0.3374
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 203s 391ms/step - accuracy: 0.8963 - loss: 0.2622 - val_accuracy: 0.8781 - val_loss: 0.3099
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 197s 393ms/step - accuracy: 0.9070 - loss: 0.2360 - val_accuracy: 0.8656 - val_loss: 0.3252


In [24]:
loss, accuracy = model.evaluate(X_test, Y_test)

1250/1250 ━━━━━━━━━━━━━━━━━━━━ 153s 122ms/step - accuracy: 0.9220 - loss: 0.2071


In [25]:
print(loss)
print(accuracy)

0.23023664951324463
0.9110749959945679


In [26]:
def predictive_system(review):
  sequences = tokenizer.texts_to_sequences([review])
  padded_sequences = pad_sequences(sequences, maxlen=200)
  prediction = model.predict(padded_sequences)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [27]:
predictive_system("I loved this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step


'positive'

In [28]:
predictive_system("I hated this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step


'negative'

In [29]:
predictive_system("I liked this movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step


'positive'

In [35]:
model.save("model.h5")

from google.colab import files
files.download("model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

from google.colab import files
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>